In [62]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import dask.dataframe as dd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

In [63]:
# Set index to be reindex to
new_lat = np.arange(-90, 90, 0.5)
new_lon = np.arange(-180, 180, 0.5)

In [ ]:
# Opening all NMME files
CanESM5 = xr.open_mfdataset('data/NMME/CanESM5/prec/*.nc')
CanESM5 = CanESM5.assign_coords(X=(((CanESM5.X + 180) % 360) - 180)).sortby(['X'])
CanESM5 = CanESM5.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

CCSM4 = xr.open_mfdataset('data/NMME/COLA-RSMAS-CCSM4/prec/*.nc')
CCSM4 = CCSM4.assign_coords(X=(((CCSM4.X + 180) % 360) - 180)).sortby(['X'])
CCSM4 = CCSM4.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

CESM1 = xr.open_mfdataset('data/NMME/COLA-RSMAS-CESM1/prec/*.nc')
CESM1 = CESM1.assign_coords(X=(((CESM1.X + 180) % 360) - 180)).sortby(['X'])
CESM1 = CESM1.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

GEM5 = xr.open_mfdataset('data/NMME/GEM5.2-NEMO/prec/*.nc')
GEM5 = GEM5.assign_coords(X=(((GEM5.X + 180) % 360) - 180)).sortby(['X'])
GEM5 = GEM5.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

GFDL = xr.open_mfdataset('data/NMME/GFDL-SPEAR/prec/*.nc')
GFDL = GFDL.assign_coords(X=(((GFDL.X + 180) % 360) - 180)).sortby(['X'])
GFDL = GFDL.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

NASA = xr.open_mfdataset('data/NMME/NASA-GEOSS2S/prec/*.nc')
NASA = NASA.assign_coords(X=(((NASA.X + 180) % 360) - 180)).sortby(['X'])
NASA = NASA.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

NCEP = xr.open_mfdataset('data/NMME/NCEP-CFSv2/prec/*.nc')
NCEP = NCEP.assign_coords(X=(((NCEP.X + 180) % 360) - 180)).sortby(['X'])
NCEP = NCEP.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

In [36]:
# Opening all CDS files
CMCC = xr.open_mfdataset('data/CDS/CMCC/prec/*.nc')
CMCC = CMCC.assign_coords(X=(((CMCC.X + 180) % 360) - 180)).sortby(['X'])
CMCC = CMCC.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

DWD = xr.open_mfdataset('data/CDS/DWD/prec/*.nc')
DWD = DWD.assign_coords(X=(((DWD.X + 180) % 360) - 180)).sortby(['X'])
DWD = DWD.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

METEO = xr.open_mfdataset('data/CDS/METEO_FRANCE/prec/*.nc')
METEO = METEO.assign_coords(X=(((METEO.X + 180) % 360) - 180)).sortby(['X'])
METEO = METEO.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

ECMWF = xr.open_mfdataset('data/CDS/ECMWF/prec/*.nc')
ECMWF = ECMWF.assign_coords(X=(((ECMWF.X + 180) % 360) - 180)).sortby(['X'])
ECMWF = ECMWF.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

In [34]:
# Opening CHIRPS data
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [35]:
# Subset the South Sudan Region CanESM5
CanESM5_south_sudan = (CanESM5.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(CanESM5_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for CanESM5
CanESM5_south_sudan_df = CanESM5_south_sudan.to_dataframe().reset_index()
CanESM5_south_sudan_df['realization time'] = CanESM5_south_sudan_df['date of prediction'] + (
            CanESM5_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_south_sudan_df['month'] = CanESM5_south_sudan_df['realization time'].dt.month
CanESM5_south_sudan_df['year'] = CanESM5_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
south_sudan_merged_df = CanESM5_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_CanESM5_merged.nc')


# Subset the Eastern East Africa Region CanESM5
CanESM5_eastern_east_africa = (CanESM5.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of CanESM5
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    CanESM5_eastern_east_africa, method='nearest')

# Calculate realized dates for CanESM5
CanESM5_eastern_east_africa_df = CanESM5_eastern_east_africa.to_dataframe().reset_index()
CanESM5_eastern_east_africa_df['realization time'] = CanESM5_eastern_east_africa_df['date of prediction'] + (
            CanESM5_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_eastern_east_africa_df['month'] = CanESM5_eastern_east_africa_df['realization time'].dt.month
CanESM5_eastern_east_africa_df['year'] = CanESM5_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
eastern_east_africa_merged_df = CanESM5_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_CanESM5_merged.nc')

# Subset the Eastern Ukraine Region CanESM5
CanESM5_eastern_ukraine = (CanESM5.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    CanESM5_eastern_ukraine, method='nearest')

# Calculate realized dates for CanESM5
CanESM5_eastern_ukraine_df = CanESM5_eastern_ukraine.to_dataframe().reset_index()
CanESM5_eastern_ukraine_df['realization time'] = CanESM5_eastern_ukraine_df['date of prediction'] + (
            CanESM5_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_eastern_ukraine_df['month'] = CanESM5_eastern_ukraine_df['realization time'].dt.month
CanESM5_eastern_ukraine_df['year'] = CanESM5_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
eastern_ukraine_merged_df = CanESM5_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_CanESM5_merged.nc')


# Subset the Southern Africa Region CanESM5
CanESM5_southern_africa = (CanESM5.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    CanESM5_southern_africa, method='nearest')

# Calculate realized dates for CanESM5
CanESM5_southern_africa_df = CanESM5_southern_africa.to_dataframe().reset_index()
CanESM5_southern_africa_df['realization time'] = CanESM5_southern_africa_df['date of prediction'] + (
            CanESM5_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_southern_africa_df['month'] = CanESM5_southern_africa_df['realization time'].dt.month
CanESM5_southern_africa_df['year'] = CanESM5_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
southern_africa_merged_df = CanESM5_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_CanESM5_merged.nc')


# Subset the West Africa Region CanESM5
CanESM5_west_africa = (CanESM5.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(CanESM5_west_africa, method='nearest'))

# Calculate realized dates for CanESM5
CanESM5_west_africa_df = CanESM5_west_africa.to_dataframe().reset_index()
CanESM5_west_africa_df['realization time'] = CanESM5_west_africa_df['date of prediction'] + (
            CanESM5_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_west_africa_df['month'] = CanESM5_west_africa_df['realization time'].dt.month
CanESM5_west_africa_df['year'] = CanESM5_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
west_africa_merged_df = CanESM5_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_CanESM5_merged.nc')


# Subset the Sri Lanka Region CanESM5
CanESM5_sri_lanka = (CanESM5.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(CanESM5_sri_lanka, method='nearest'))

# Calculate realized dates for CanESM5
CanESM5_sri_lanka_df = CanESM5_sri_lanka.to_dataframe().reset_index()
CanESM5_sri_lanka_df['realization time'] = CanESM5_sri_lanka_df['date of prediction'] + (
            CanESM5_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_sri_lanka_df['month'] = CanESM5_sri_lanka_df['realization time'].dt.month
CanESM5_sri_lanka_df['year'] = CanESM5_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
sri_lanka_merged_df = CanESM5_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_CanESM5_merged.nc')


# Subset the Lake Victoria Basin Region CanESM5
CanESM5_lake_victoria_basin = (CanESM5.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(CanESM5_lake_victoria_basin, method='nearest'))

# Calculate realized dates for CanESM5
CanESM5_lake_victoria_basin_df = CanESM5_lake_victoria_basin.to_dataframe().reset_index()
CanESM5_lake_victoria_basin_df['realization time'] = CanESM5_lake_victoria_basin_df['date of prediction'] + (
            CanESM5_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
CanESM5_lake_victoria_basin_df['month'] = CanESM5_lake_victoria_basin_df['realization time'].dt.month
CanESM5_lake_victoria_basin_df['year'] = CanESM5_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and CanESM5 dataframe
lake_victoria_basin_merged_df = CanESM5_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_CanESM5_merged.nc')


KeyboardInterrupt



In [ ]:
# Subset the South Sudan Region CCSM4
CCSM4_south_sudan = (CCSM4.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(CCSM4_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for CCSM4
CCSM4_south_sudan_df = CCSM4_south_sudan.to_dataframe().reset_index()
CCSM4_south_sudan_df['realization time'] = CCSM4_south_sudan_df['date of prediction'] + (
            CCSM4_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_south_sudan_df['month'] = CCSM4_south_sudan_df['realization time'].dt.month
CCSM4_south_sudan_df['year'] = CCSM4_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
south_sudan_merged_df = CCSM4_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_CCSM4_merged.nc')


# Subset the Eastern East Africa Region CCSM4
CCSM4_eastern_east_africa = (CCSM4.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of CCSM4
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    CCSM4_eastern_east_africa, method='nearest')

# Calculate realized dates for CCSM4
CCSM4_eastern_east_africa_df = CCSM4_eastern_east_africa.to_dataframe().reset_index()
CCSM4_eastern_east_africa_df['realization time'] = CCSM4_eastern_east_africa_df['date of prediction'] + (
            CCSM4_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_eastern_east_africa_df['month'] = CCSM4_eastern_east_africa_df['realization time'].dt.month
CCSM4_eastern_east_africa_df['year'] = CCSM4_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
eastern_east_africa_merged_df = CCSM4_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_CCSM4_merged.nc')

# Subset the Eastern Ukraine Region CCSM4
CCSM4_eastern_ukraine = (CCSM4.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    CCSM4_eastern_ukraine, method='nearest')

# Calculate realized dates for CCSM4
CCSM4_eastern_ukraine_df = CCSM4_eastern_ukraine.to_dataframe().reset_index()
CCSM4_eastern_ukraine_df['realization time'] = CCSM4_eastern_ukraine_df['date of prediction'] + (
            CCSM4_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_eastern_ukraine_df['month'] = CCSM4_eastern_ukraine_df['realization time'].dt.month
CCSM4_eastern_ukraine_df['year'] = CCSM4_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
eastern_ukraine_merged_df = CCSM4_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_CCSM4_merged.nc')


# Subset the Southern Africa Region CCSM4
CCSM4_southern_africa = (CCSM4.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    CCSM4_southern_africa, method='nearest')

# Calculate realized dates for CCSM4
CCSM4_southern_africa_df = CCSM4_southern_africa.to_dataframe().reset_index()
CCSM4_southern_africa_df['realization time'] = CCSM4_southern_africa_df['date of prediction'] + (
            CCSM4_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_southern_africa_df['month'] = CCSM4_southern_africa_df['realization time'].dt.month
CCSM4_southern_africa_df['year'] = CCSM4_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
southern_africa_merged_df = CCSM4_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_CCSM4_merged.nc')


# Subset the West Africa Region CCSM4
CCSM4_west_africa = (CCSM4.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(CCSM4_west_africa, method='nearest'))

# Calculate realized dates for CCSM4
CCSM4_west_africa_df = CCSM4_west_africa.to_dataframe().reset_index()
CCSM4_west_africa_df['realization time'] = CCSM4_west_africa_df['date of prediction'] + (
            CCSM4_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_west_africa_df['month'] = CCSM4_west_africa_df['realization time'].dt.month
CCSM4_west_africa_df['year'] = CCSM4_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
west_africa_merged_df = CCSM4_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_CCSM4_merged.nc')


# Subset the Sri Lanka Region CCSM4
CCSM4_sri_lanka = (CCSM4.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(CCSM4_sri_lanka, method='nearest'))

# Calculate realized dates for CCSM4
CCSM4_sri_lanka_df = CCSM4_sri_lanka.to_dataframe().reset_index()
CCSM4_sri_lanka_df['realization time'] = CCSM4_sri_lanka_df['date of prediction'] + (
            CCSM4_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_sri_lanka_df['month'] = CCSM4_sri_lanka_df['realization time'].dt.month
CCSM4_sri_lanka_df['year'] = CCSM4_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
sri_lanka_merged_df = CCSM4_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_CCSM4_merged.nc')


# Subset the Lake Victoria Basin Region CCSM4
CCSM4_lake_victoria_basin = (CCSM4.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(CCSM4_lake_victoria_basin, method='nearest'))

# Calculate realized dates for CCSM4
CCSM4_lake_victoria_basin_df = CCSM4_lake_victoria_basin.to_dataframe().reset_index()
CCSM4_lake_victoria_basin_df['realization time'] = CCSM4_lake_victoria_basin_df['date of prediction'] + (
            CCSM4_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
CCSM4_lake_victoria_basin_df['month'] = CCSM4_lake_victoria_basin_df['realization time'].dt.month
CCSM4_lake_victoria_basin_df['year'] = CCSM4_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and CCSM4 dataframe
lake_victoria_basin_merged_df = CCSM4_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_CCSM4_merged.nc')

In [ ]:
# Subset the South Sudan Region CESM1
CESM1_south_sudan = (CESM1.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(CESM1_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for CESM1
CESM1_south_sudan_df = CESM1_south_sudan.to_dataframe().reset_index()
CESM1_south_sudan_df['realization time'] = CESM1_south_sudan_df['date of prediction'] + (
            CESM1_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_south_sudan_df['month'] = CESM1_south_sudan_df['realization time'].dt.month
CESM1_south_sudan_df['year'] = CESM1_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
south_sudan_merged_df = CESM1_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_CESM1_merged.nc')


# Subset the Eastern East Africa Region CESM1
CESM1_eastern_east_africa = (CESM1.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of CESM1
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    CESM1_eastern_east_africa, method='nearest')

# Calculate realized dates for CESM1
CESM1_eastern_east_africa_df = CESM1_eastern_east_africa.to_dataframe().reset_index()
CESM1_eastern_east_africa_df['realization time'] = CESM1_eastern_east_africa_df['date of prediction'] + (
            CESM1_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_eastern_east_africa_df['month'] = CESM1_eastern_east_africa_df['realization time'].dt.month
CESM1_eastern_east_africa_df['year'] = CESM1_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
eastern_east_africa_merged_df = CESM1_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_CESM1_merged.nc')

# Subset the Eastern Ukraine Region CESM1
CESM1_eastern_ukraine = (CESM1.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    CESM1_eastern_ukraine, method='nearest')

# Calculate realized dates for CESM1
CESM1_eastern_ukraine_df = CESM1_eastern_ukraine.to_dataframe().reset_index()
CESM1_eastern_ukraine_df['realization time'] = CESM1_eastern_ukraine_df['date of prediction'] + (
            CESM1_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_eastern_ukraine_df['month'] = CESM1_eastern_ukraine_df['realization time'].dt.month
CESM1_eastern_ukraine_df['year'] = CESM1_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
eastern_ukraine_merged_df = CESM1_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_CESM1_merged.nc')


# Subset the Southern Africa Region CESM1
CESM1_southern_africa = (CESM1.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    CESM1_southern_africa, method='nearest')

# Calculate realized dates for CESM1
CESM1_southern_africa_df = CESM1_southern_africa.to_dataframe().reset_index()
CESM1_southern_africa_df['realization time'] = CESM1_southern_africa_df['date of prediction'] + (
            CESM1_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_southern_africa_df['month'] = CESM1_southern_africa_df['realization time'].dt.month
CESM1_southern_africa_df['year'] = CESM1_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
southern_africa_merged_df = CESM1_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_CESM1_merged.nc')


# Subset the West Africa Region CESM1
CESM1_west_africa = (CESM1.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(CESM1_west_africa, method='nearest'))

# Calculate realized dates for CESM1
CESM1_west_africa_df = CESM1_west_africa.to_dataframe().reset_index()
CESM1_west_africa_df['realization time'] = CESM1_west_africa_df['date of prediction'] + (
            CESM1_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_west_africa_df['month'] = CESM1_west_africa_df['realization time'].dt.month
CESM1_west_africa_df['year'] = CESM1_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
west_africa_merged_df = CESM1_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_CESM1_merged.nc')


# Subset the Sri Lanka Region CESM1
CESM1_sri_lanka = (CESM1.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(CESM1_sri_lanka, method='nearest'))

# Calculate realized dates for CESM1
CESM1_sri_lanka_df = CESM1_sri_lanka.to_dataframe().reset_index()
CESM1_sri_lanka_df['realization time'] = CESM1_sri_lanka_df['date of prediction'] + (
            CESM1_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_sri_lanka_df['month'] = CESM1_sri_lanka_df['realization time'].dt.month
CESM1_sri_lanka_df['year'] = CESM1_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
sri_lanka_merged_df = CESM1_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_CESM1_merged.nc')


# Subset the Lake Victoria Basin Region CESM1
CESM1_lake_victoria_basin = (CESM1.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(CESM1_lake_victoria_basin, method='nearest'))

# Calculate realized dates for CESM1
CESM1_lake_victoria_basin_df = CESM1_lake_victoria_basin.to_dataframe().reset_index()
CESM1_lake_victoria_basin_df['realization time'] = CESM1_lake_victoria_basin_df['date of prediction'] + (
            CESM1_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
CESM1_lake_victoria_basin_df['month'] = CESM1_lake_victoria_basin_df['realization time'].dt.month
CESM1_lake_victoria_basin_df['year'] = CESM1_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and CESM1 dataframe
lake_victoria_basin_merged_df = CESM1_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_CESM1_merged.nc')

In [ ]:
# Subset the South Sudan Region GEM5
GEM5_south_sudan = (GEM5.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(GEM5_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for GEM5
GEM5_south_sudan_df = GEM5_south_sudan.to_dataframe().reset_index()
GEM5_south_sudan_df['realization time'] = GEM5_south_sudan_df['date of prediction'] + (
            GEM5_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_south_sudan_df['month'] = GEM5_south_sudan_df['realization time'].dt.month
GEM5_south_sudan_df['year'] = GEM5_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
south_sudan_merged_df = GEM5_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_GEM5_merged.nc')


# Subset the Eastern East Africa Region GEM5
GEM5_eastern_east_africa = (GEM5.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of GEM5
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    GEM5_eastern_east_africa, method='nearest')

# Calculate realized dates for GEM5
GEM5_eastern_east_africa_df = GEM5_eastern_east_africa.to_dataframe().reset_index()
GEM5_eastern_east_africa_df['realization time'] = GEM5_eastern_east_africa_df['date of prediction'] + (
            GEM5_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_eastern_east_africa_df['month'] = GEM5_eastern_east_africa_df['realization time'].dt.month
GEM5_eastern_east_africa_df['year'] = GEM5_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
eastern_east_africa_merged_df = GEM5_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_GEM5_merged.nc')

# Subset the Eastern Ukraine Region GEM5
GEM5_eastern_ukraine = (GEM5.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    GEM5_eastern_ukraine, method='nearest')

# Calculate realized dates for GEM5
GEM5_eastern_ukraine_df = GEM5_eastern_ukraine.to_dataframe().reset_index()
GEM5_eastern_ukraine_df['realization time'] = GEM5_eastern_ukraine_df['date of prediction'] + (
            GEM5_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_eastern_ukraine_df['month'] = GEM5_eastern_ukraine_df['realization time'].dt.month
GEM5_eastern_ukraine_df['year'] = GEM5_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
eastern_ukraine_merged_df = GEM5_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_GEM5_merged.nc')


# Subset the Southern Africa Region GEM5
GEM5_southern_africa = (GEM5.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    GEM5_southern_africa, method='nearest')

# Calculate realized dates for GEM5
GEM5_southern_africa_df = GEM5_southern_africa.to_dataframe().reset_index()
GEM5_southern_africa_df['realization time'] = GEM5_southern_africa_df['date of prediction'] + (
            GEM5_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_southern_africa_df['month'] = GEM5_southern_africa_df['realization time'].dt.month
GEM5_southern_africa_df['year'] = GEM5_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
southern_africa_merged_df = GEM5_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_GEM5_merged.nc')


# Subset the West Africa Region GEM5
GEM5_west_africa = (GEM5.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(GEM5_west_africa, method='nearest'))

# Calculate realized dates for GEM5
GEM5_west_africa_df = GEM5_west_africa.to_dataframe().reset_index()
GEM5_west_africa_df['realization time'] = GEM5_west_africa_df['date of prediction'] + (
            GEM5_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_west_africa_df['month'] = GEM5_west_africa_df['realization time'].dt.month
GEM5_west_africa_df['year'] = GEM5_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
west_africa_merged_df = GEM5_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_GEM5_merged.nc')


# Subset the Sri Lanka Region GEM5
GEM5_sri_lanka = (GEM5.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(GEM5_sri_lanka, method='nearest'))

# Calculate realized dates for GEM5
GEM5_sri_lanka_df = GEM5_sri_lanka.to_dataframe().reset_index()
GEM5_sri_lanka_df['realization time'] = GEM5_sri_lanka_df['date of prediction'] + (
            GEM5_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_sri_lanka_df['month'] = GEM5_sri_lanka_df['realization time'].dt.month
GEM5_sri_lanka_df['year'] = GEM5_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
sri_lanka_merged_df = GEM5_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_GEM5_merged.nc')


# Subset the Lake Victoria Basin Region GEM5
GEM5_lake_victoria_basin = (GEM5.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(GEM5_lake_victoria_basin, method='nearest'))

# Calculate realized dates for GEM5
GEM5_lake_victoria_basin_df = GEM5_lake_victoria_basin.to_dataframe().reset_index()
GEM5_lake_victoria_basin_df['realization time'] = GEM5_lake_victoria_basin_df['date of prediction'] + (
            GEM5_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
GEM5_lake_victoria_basin_df['month'] = GEM5_lake_victoria_basin_df['realization time'].dt.month
GEM5_lake_victoria_basin_df['year'] = GEM5_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and GEM5 dataframe
lake_victoria_basin_merged_df = GEM5_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_GEM5_merged.nc')

In [ ]:
# Subset the South Sudan Region GFDL
GFDL_south_sudan = (GFDL.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(GFDL_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for GFDL
GFDL_south_sudan_df = GFDL_south_sudan.to_dataframe().reset_index()
GFDL_south_sudan_df['realization time'] = GFDL_south_sudan_df['date of prediction'] + (
            GFDL_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_south_sudan_df['month'] = GFDL_south_sudan_df['realization time'].dt.month
GFDL_south_sudan_df['year'] = GFDL_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
south_sudan_merged_df = GFDL_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_GFDL_merged.nc')


# Subset the Eastern East Africa Region GFDL
GFDL_eastern_east_africa = (GFDL.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of GFDL
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    GFDL_eastern_east_africa, method='nearest')

# Calculate realized dates for GFDL
GFDL_eastern_east_africa_df = GFDL_eastern_east_africa.to_dataframe().reset_index()
GFDL_eastern_east_africa_df['realization time'] = GFDL_eastern_east_africa_df['date of prediction'] + (
            GFDL_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_eastern_east_africa_df['month'] = GFDL_eastern_east_africa_df['realization time'].dt.month
GFDL_eastern_east_africa_df['year'] = GFDL_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
eastern_east_africa_merged_df = GFDL_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_GFDL_merged.nc')

# Subset the Eastern Ukraine Region GFDL
GFDL_eastern_ukraine = (GFDL.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    GFDL_eastern_ukraine, method='nearest')

# Calculate realized dates for GFDL
GFDL_eastern_ukraine_df = GFDL_eastern_ukraine.to_dataframe().reset_index()
GFDL_eastern_ukraine_df['realization time'] = GFDL_eastern_ukraine_df['date of prediction'] + (
            GFDL_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_eastern_ukraine_df['month'] = GFDL_eastern_ukraine_df['realization time'].dt.month
GFDL_eastern_ukraine_df['year'] = GFDL_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
eastern_ukraine_merged_df = GFDL_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_GFDL_merged.nc')


# Subset the Southern Africa Region GFDL
GFDL_southern_africa = (GFDL.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    GFDL_southern_africa, method='nearest')

# Calculate realized dates for GFDL
GFDL_southern_africa_df = GFDL_southern_africa.to_dataframe().reset_index()
GFDL_southern_africa_df['realization time'] = GFDL_southern_africa_df['date of prediction'] + (
            GFDL_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_southern_africa_df['month'] = GFDL_southern_africa_df['realization time'].dt.month
GFDL_southern_africa_df['year'] = GFDL_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
southern_africa_merged_df = GFDL_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_GFDL_merged.nc')


# Subset the West Africa Region GFDL
GFDL_west_africa = (GFDL.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(GFDL_west_africa, method='nearest'))

# Calculate realized dates for GFDL
GFDL_west_africa_df = GFDL_west_africa.to_dataframe().reset_index()
GFDL_west_africa_df['realization time'] = GFDL_west_africa_df['date of prediction'] + (
            GFDL_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_west_africa_df['month'] = GFDL_west_africa_df['realization time'].dt.month
GFDL_west_africa_df['year'] = GFDL_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
west_africa_merged_df = GFDL_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_GFDL_merged.nc')


# Subset the Sri Lanka Region GFDL
GFDL_sri_lanka = (GFDL.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(GFDL_sri_lanka, method='nearest'))

# Calculate realized dates for GFDL
GFDL_sri_lanka_df = GFDL_sri_lanka.to_dataframe().reset_index()
GFDL_sri_lanka_df['realization time'] = GFDL_sri_lanka_df['date of prediction'] + (
            GFDL_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_sri_lanka_df['month'] = GFDL_sri_lanka_df['realization time'].dt.month
GFDL_sri_lanka_df['year'] = GFDL_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
sri_lanka_merged_df = GFDL_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_GFDL_merged.nc')


# Subset the Lake Victoria Basin Region GFDL
GFDL_lake_victoria_basin = (GFDL.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(GFDL_lake_victoria_basin, method='nearest'))

# Calculate realized dates for GFDL
GFDL_lake_victoria_basin_df = GFDL_lake_victoria_basin.to_dataframe().reset_index()
GFDL_lake_victoria_basin_df['realization time'] = GFDL_lake_victoria_basin_df['date of prediction'] + (
            GFDL_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_lake_victoria_basin_df['month'] = GFDL_lake_victoria_basin_df['realization time'].dt.month
GFDL_lake_victoria_basin_df['year'] = GFDL_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and GFDL dataframe
lake_victoria_basin_merged_df = GFDL_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_GFDL_merged.nc')

In [ ]:
# Subset the South Sudan Region NASA
NASA_south_sudan = (NASA.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(NASA_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for NASA
NASA_south_sudan_df = NASA_south_sudan.to_dataframe().reset_index()
NASA_south_sudan_df['realization time'] = NASA_south_sudan_df['date of prediction'] + (
            NASA_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_south_sudan_df['month'] = NASA_south_sudan_df['realization time'].dt.month
NASA_south_sudan_df['year'] = NASA_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
south_sudan_merged_df = NASA_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_NASA_merged.nc')


# Subset the Eastern East Africa Region NASA
NASA_eastern_east_africa = (NASA.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NASA
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    NASA_eastern_east_africa, method='nearest')

# Calculate realized dates for NASA
NASA_eastern_east_africa_df = NASA_eastern_east_africa.to_dataframe().reset_index()
NASA_eastern_east_africa_df['realization time'] = NASA_eastern_east_africa_df['date of prediction'] + (
            NASA_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_eastern_east_africa_df['month'] = NASA_eastern_east_africa_df['realization time'].dt.month
NASA_eastern_east_africa_df['year'] = NASA_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
eastern_east_africa_merged_df = NASA_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_NASA_merged.nc')

# Subset the Eastern Ukraine Region NASA
NASA_eastern_ukraine = (NASA.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    NASA_eastern_ukraine, method='nearest')

# Calculate realized dates for NASA
NASA_eastern_ukraine_df = NASA_eastern_ukraine.to_dataframe().reset_index()
NASA_eastern_ukraine_df['realization time'] = NASA_eastern_ukraine_df['date of prediction'] + (
            NASA_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_eastern_ukraine_df['month'] = NASA_eastern_ukraine_df['realization time'].dt.month
NASA_eastern_ukraine_df['year'] = NASA_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
eastern_ukraine_merged_df = NASA_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_NASA_merged.nc')


# Subset the Southern Africa Region NASA
NASA_southern_africa = (NASA.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    NASA_southern_africa, method='nearest')

# Calculate realized dates for NASA
NASA_southern_africa_df = NASA_southern_africa.to_dataframe().reset_index()
NASA_southern_africa_df['realization time'] = NASA_southern_africa_df['date of prediction'] + (
            NASA_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_southern_africa_df['month'] = NASA_southern_africa_df['realization time'].dt.month
NASA_southern_africa_df['year'] = NASA_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
southern_africa_merged_df = NASA_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_NASA_merged.nc')


# Subset the West Africa Region NASA
NASA_west_africa = (NASA.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(NASA_west_africa, method='nearest'))

# Calculate realized dates for NASA
NASA_west_africa_df = NASA_west_africa.to_dataframe().reset_index()
NASA_west_africa_df['realization time'] = NASA_west_africa_df['date of prediction'] + (
            NASA_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_west_africa_df['month'] = NASA_west_africa_df['realization time'].dt.month
NASA_west_africa_df['year'] = NASA_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
west_africa_merged_df = NASA_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_NASA_merged.nc')


# Subset the Sri Lanka Region NASA
NASA_sri_lanka = (NASA.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(NASA_sri_lanka, method='nearest'))

# Calculate realized dates for NASA
NASA_sri_lanka_df = NASA_sri_lanka.to_dataframe().reset_index()
NASA_sri_lanka_df['realization time'] = NASA_sri_lanka_df['date of prediction'] + (
            NASA_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_sri_lanka_df['month'] = NASA_sri_lanka_df['realization time'].dt.month
NASA_sri_lanka_df['year'] = NASA_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
sri_lanka_merged_df = NASA_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_NASA_merged.nc')


# Subset the Lake Victoria Basin Region NASA
NASA_lake_victoria_basin = (NASA.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(NASA_lake_victoria_basin, method='nearest'))

# Calculate realized dates for NASA
NASA_lake_victoria_basin_df = NASA_lake_victoria_basin.to_dataframe().reset_index()
NASA_lake_victoria_basin_df['realization time'] = NASA_lake_victoria_basin_df['date of prediction'] + (
            NASA_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
NASA_lake_victoria_basin_df['month'] = NASA_lake_victoria_basin_df['realization time'].dt.month
NASA_lake_victoria_basin_df['year'] = NASA_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and NASA dataframe
lake_victoria_basin_merged_df = NASA_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_NASA_merged.nc')

In [ ]:
# Subset the South Sudan Region NCEP
NCEP_south_sudan = (NCEP.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(NCEP_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for NCEP
NCEP_south_sudan_df = NCEP_south_sudan.to_dataframe().reset_index()
NCEP_south_sudan_df['realization time'] = NCEP_south_sudan_df['date of prediction'] + (
            NCEP_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_south_sudan_df['month'] = NCEP_south_sudan_df['realization time'].dt.month
NCEP_south_sudan_df['year'] = NCEP_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
south_sudan_merged_df = NCEP_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_NCEP_merged.nc')


# Subset the Eastern East Africa Region NCEP
NCEP_eastern_east_africa = (NCEP.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NCEP
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    NCEP_eastern_east_africa, method='nearest')

# Calculate realized dates for NCEP
NCEP_eastern_east_africa_df = NCEP_eastern_east_africa.to_dataframe().reset_index()
NCEP_eastern_east_africa_df['realization time'] = NCEP_eastern_east_africa_df['date of prediction'] + (
            NCEP_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_eastern_east_africa_df['month'] = NCEP_eastern_east_africa_df['realization time'].dt.month
NCEP_eastern_east_africa_df['year'] = NCEP_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
eastern_east_africa_merged_df = NCEP_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_NCEP_merged.nc')

# Subset the Eastern Ukraine Region NCEP
NCEP_eastern_ukraine = (NCEP.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    NCEP_eastern_ukraine, method='nearest')

# Calculate realized dates for NCEP
NCEP_eastern_ukraine_df = NCEP_eastern_ukraine.to_dataframe().reset_index()
NCEP_eastern_ukraine_df['realization time'] = NCEP_eastern_ukraine_df['date of prediction'] + (
            NCEP_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_eastern_ukraine_df['month'] = NCEP_eastern_ukraine_df['realization time'].dt.month
NCEP_eastern_ukraine_df['year'] = NCEP_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
eastern_ukraine_merged_df = NCEP_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_NCEP_merged.nc')


# Subset the Southern Africa Region NCEP
NCEP_southern_africa = (NCEP.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    NCEP_southern_africa, method='nearest')

# Calculate realized dates for NCEP
NCEP_southern_africa_df = NCEP_southern_africa.to_dataframe().reset_index()
NCEP_southern_africa_df['realization time'] = NCEP_southern_africa_df['date of prediction'] + (
            NCEP_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_southern_africa_df['month'] = NCEP_southern_africa_df['realization time'].dt.month
NCEP_southern_africa_df['year'] = NCEP_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
southern_africa_merged_df = NCEP_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_NCEP_merged.nc')


# Subset the West Africa Region NCEP
NCEP_west_africa = (NCEP.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(NCEP_west_africa, method='nearest'))

# Calculate realized dates for NCEP
NCEP_west_africa_df = NCEP_west_africa.to_dataframe().reset_index()
NCEP_west_africa_df['realization time'] = NCEP_west_africa_df['date of prediction'] + (
            NCEP_west_africa_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_west_africa_df['month'] = NCEP_west_africa_df['realization time'].dt.month
NCEP_west_africa_df['year'] = NCEP_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
west_africa_merged_df = NCEP_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_NCEP_merged.nc')


# Subset the Sri Lanka Region NCEP
NCEP_sri_lanka = (NCEP.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(NCEP_sri_lanka, method='nearest'))

# Calculate realized dates for NCEP
NCEP_sri_lanka_df = NCEP_sri_lanka.to_dataframe().reset_index()
NCEP_sri_lanka_df['realization time'] = NCEP_sri_lanka_df['date of prediction'] + (
            NCEP_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_sri_lanka_df['month'] = NCEP_sri_lanka_df['realization time'].dt.month
NCEP_sri_lanka_df['year'] = NCEP_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
sri_lanka_merged_df = NCEP_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_NCEP_merged.nc')


# Subset the Lake Victoria Basin Region NCEP
NCEP_lake_victoria_basin = (NCEP.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(NCEP_lake_victoria_basin, method='nearest'))

# Calculate realized dates for NCEP
NCEP_lake_victoria_basin_df = NCEP_lake_victoria_basin.to_dataframe().reset_index()
NCEP_lake_victoria_basin_df['realization time'] = NCEP_lake_victoria_basin_df['date of prediction'] + (
            NCEP_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]')
NCEP_lake_victoria_basin_df['month'] = NCEP_lake_victoria_basin_df['realization time'].dt.month
NCEP_lake_victoria_basin_df['year'] = NCEP_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and NCEP dataframe
lake_victoria_basin_merged_df = NCEP_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_NCEP_merged.nc')

In [37]:
# Subset the South Sudan Region CMCC
CMCC_south_sudan = (CMCC.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(CMCC_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for CMCC
CMCC_south_sudan_df = CMCC_south_sudan.to_dataframe().reset_index()
CMCC_south_sudan_df['lead_time'] = CMCC_south_sudan_df['lead_time'] - 0.5
CMCC_south_sudan_df['realization time'] = CMCC_south_sudan_df['date of prediction'] + (
            CMCC_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_south_sudan_df['month'] = CMCC_south_sudan_df['realization time'].dt.month
CMCC_south_sudan_df['year'] = CMCC_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
south_sudan_merged_df = CMCC_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_CMCC_merged.nc')


# Subset the Eastern East Africa Region CMCC
CMCC_eastern_east_africa = (CMCC.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of CMCC
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    CMCC_eastern_east_africa, method='nearest')

# Calculate realized dates for CMCC
CMCC_eastern_east_africa_df = CMCC_eastern_east_africa.to_dataframe().reset_index()
CMCC_eastern_east_africa_df['lead_time'] = CMCC_eastern_east_africa_df['lead_time'] - 0.5
CMCC_eastern_east_africa_df['realization time'] = CMCC_eastern_east_africa_df['date of prediction'] + (
            CMCC_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_eastern_east_africa_df['month'] = CMCC_eastern_east_africa_df['realization time'].dt.month
CMCC_eastern_east_africa_df['year'] = CMCC_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
eastern_east_africa_merged_df = CMCC_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_CMCC_merged.nc')

# Subset the Eastern Ukraine Region CMCC
CMCC_eastern_ukraine = (CMCC.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    CMCC_eastern_ukraine, method='nearest')

# Calculate realized dates for CMCC
CMCC_eastern_ukraine_df = CMCC_eastern_ukraine.to_dataframe().reset_index()
CMCC_eastern_ukraine_df['lead_time'] = CMCC_eastern_ukraine_df['lead_time'] - 0.5
CMCC_eastern_ukraine_df['realization time'] = CMCC_eastern_ukraine_df['date of prediction'] + (
            CMCC_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_eastern_ukraine_df['month'] = CMCC_eastern_ukraine_df['realization time'].dt.month
CMCC_eastern_ukraine_df['year'] = CMCC_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
eastern_ukraine_merged_df = CMCC_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_CMCC_merged.nc')


# Subset the Southern Africa Region CMCC
CMCC_southern_africa = (CMCC.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    CMCC_southern_africa, method='nearest')

# Calculate realized dates for CMCC
CMCC_southern_africa_df = CMCC_southern_africa.to_dataframe().reset_index()
CMCC_southern_africa_df['lead_time'] = CMCC_southern_africa_df['lead_time'] - 0.5
CMCC_southern_africa_df['realization time'] = CMCC_southern_africa_df['date of prediction'] + (
            CMCC_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_southern_africa_df['month'] = CMCC_southern_africa_df['realization time'].dt.month
CMCC_southern_africa_df['year'] = CMCC_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
southern_africa_merged_df = CMCC_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_CMCC_merged.nc')


# Subset the West Africa Region CMCC
CMCC_west_africa = (CMCC.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(CMCC_west_africa, method='nearest'))

# Calculate realized dates for CMCC
CMCC_west_africa_df = CMCC_west_africa.to_dataframe().reset_index()
CMCC_west_africa_df['lead_time'] = CMCC_west_africa_df['lead_time'] - 0.5
CMCC_west_africa_df['realization time'] = CMCC_west_africa_df['date of prediction'] + (
            CMCC_west_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_west_africa_df['month'] = CMCC_west_africa_df['realization time'].dt.month
CMCC_west_africa_df['year'] = CMCC_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
west_africa_merged_df = CMCC_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_CMCC_merged.nc')


# Subset the Sri Lanka Region CMCC
CMCC_sri_lanka = (CMCC.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(CMCC_sri_lanka, method='nearest'))

# Calculate realized dates for CMCC
CMCC_sri_lanka_df = CMCC_sri_lanka.to_dataframe().reset_index()
CMCC_sri_lanka_df['lead_time'] = CMCC_sri_lanka_df['lead_time'] - 0.5
CMCC_sri_lanka_df['realization time'] = CMCC_sri_lanka_df['date of prediction'] + (
            CMCC_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_sri_lanka_df['month'] = CMCC_sri_lanka_df['realization time'].dt.month
CMCC_sri_lanka_df['year'] = CMCC_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
sri_lanka_merged_df = CMCC_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_CMCC_merged.nc')


# Subset the Lake Victoria Basin Region CMCC
CMCC_lake_victoria_basin = (CMCC.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(CMCC_lake_victoria_basin, method='nearest'))

# Calculate realized dates for CMCC
CMCC_lake_victoria_basin_df = CMCC_lake_victoria_basin.to_dataframe().reset_index()
CMCC_lake_victoria_basin_df['lead_time'] = CMCC_lake_victoria_basin_df['lead_time'] - 0.5
CMCC_lake_victoria_basin_df['realization time'] = CMCC_lake_victoria_basin_df['date of prediction'] + (
            CMCC_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
CMCC_lake_victoria_basin_df['month'] = CMCC_lake_victoria_basin_df['realization time'].dt.month
CMCC_lake_victoria_basin_df['year'] = CMCC_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and CMCC dataframe
lake_victoria_basin_merged_df = CMCC_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_CMCC_merged.nc')

ValueError: cannot convert a DataFrame with a non-unique MultiIndex into xarray

In [ ]:
# Subset the South Sudan Region DWD
DWD_south_sudan = (DWD.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(DWD_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for DWD
DWD_south_sudan_df = DWD_south_sudan.to_dataframe().reset_index()
DWD_south_sudan_df['lead_time'] = DWD_south_sudan_df['lead_time'] - 0.5
DWD_south_sudan_df['realization time'] = DWD_south_sudan_df['date of prediction'] + (
            DWD_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_south_sudan_df['month'] = DWD_south_sudan_df['realization time'].dt.month
DWD_south_sudan_df['year'] = DWD_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
south_sudan_merged_df = DWD_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_DWD_merged.nc')


# Subset the Eastern East Africa Region DWD
DWD_eastern_east_africa = (DWD.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of DWD
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    DWD_eastern_east_africa, method='nearest')

# Calculate realized dates for DWD
DWD_eastern_east_africa_df = DWD_eastern_east_africa.to_dataframe().reset_index()
DWD_eastern_east_africa_df['lead_time'] = DWD_eastern_east_africa_df['lead_time'] - 0.5
DWD_eastern_east_africa_df['realization time'] = DWD_eastern_east_africa_df['date of prediction'] + (
            DWD_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_eastern_east_africa_df['month'] = DWD_eastern_east_africa_df['realization time'].dt.month
DWD_eastern_east_africa_df['year'] = DWD_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
eastern_east_africa_merged_df = DWD_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_DWD_merged.nc')

# Subset the Eastern Ukraine Region DWD
DWD_eastern_ukraine = (DWD.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    DWD_eastern_ukraine, method='nearest')

# Calculate realized dates for DWD
DWD_eastern_ukraine_df = DWD_eastern_ukraine.to_dataframe().reset_index()
DWD_eastern_ukraine_df['lead_time'] = DWD_eastern_ukraine_df['lead_time'] - 0.5
DWD_eastern_ukraine_df['realization time'] = DWD_eastern_ukraine_df['date of prediction'] + (
            DWD_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_eastern_ukraine_df['month'] = DWD_eastern_ukraine_df['realization time'].dt.month
DWD_eastern_ukraine_df['year'] = DWD_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
eastern_ukraine_merged_df = DWD_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_DWD_merged.nc')


# Subset the Southern Africa Region DWD
DWD_southern_africa = (DWD.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    DWD_southern_africa, method='nearest')

# Calculate realized dates for DWD
DWD_southern_africa_df = DWD_southern_africa.to_dataframe().reset_index()
DWD_southern_africa_df['lead_time'] = DWD_southern_africa_df['lead_time'] - 0.5
DWD_southern_africa_df['realization time'] = DWD_southern_africa_df['date of prediction'] + (
            DWD_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_southern_africa_df['month'] = DWD_southern_africa_df['realization time'].dt.month
DWD_southern_africa_df['year'] = DWD_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
southern_africa_merged_df = DWD_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_DWD_merged.nc')


# Subset the West Africa Region DWD
DWD_west_africa = (DWD.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(DWD_west_africa, method='nearest'))

# Calculate realized dates for DWD
DWD_west_africa_df = DWD_west_africa.to_dataframe().reset_index()
DWD_west_africa_df['lead_time'] = DWD_west_africa_df['lead_time'] - 0.5
DWD_west_africa_df['realization time'] = DWD_west_africa_df['date of prediction'] + (
            DWD_west_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_west_africa_df['month'] = DWD_west_africa_df['realization time'].dt.month
DWD_west_africa_df['year'] = DWD_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
west_africa_merged_df = DWD_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_DWD_merged.nc')


# Subset the Sri Lanka Region DWD
DWD_sri_lanka = (DWD.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(DWD_sri_lanka, method='nearest'))

# Calculate realized dates for DWD
DWD_sri_lanka_df = DWD_sri_lanka.to_dataframe().reset_index()
DWD_sri_lanka_df['lead_time'] = DWD_sri_lanka_df['lead_time'] - 0.5
DWD_sri_lanka_df['realization time'] = DWD_sri_lanka_df['date of prediction'] + (
            DWD_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_sri_lanka_df['month'] = DWD_sri_lanka_df['realization time'].dt.month
DWD_sri_lanka_df['year'] = DWD_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
sri_lanka_merged_df = DWD_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_DWD_merged.nc')


# Subset the Lake Victoria Basin Region DWD
DWD_lake_victoria_basin = (DWD.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(DWD_lake_victoria_basin, method='nearest'))

# Calculate realized dates for DWD
DWD_lake_victoria_basin_df = DWD_lake_victoria_basin.to_dataframe().reset_index()
DWD_lake_victoria_basin_df['lead_time'] = DWD_lake_victoria_basin_df['lead_time'] - 0.5
DWD_lake_victoria_basin_df['realization time'] = DWD_lake_victoria_basin_df['date of prediction'] + (
            DWD_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
DWD_lake_victoria_basin_df['month'] = DWD_lake_victoria_basin_df['realization time'].dt.month
DWD_lake_victoria_basin_df['year'] = DWD_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and DWD dataframe
lake_victoria_basin_merged_df = DWD_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_DWD_merged.nc')

In [ ]:
# Subset the South Sudan Region ECMWF
ECMWF_south_sudan = (ECMWF.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(ECMWF_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for ECMWF
ECMWF_south_sudan_df = ECMWF_south_sudan.to_dataframe().reset_index()
ECMWF_south_sudan_df['lead_time'] = ECMWF_south_sudan_df['lead_time'] - 0.5
ECMWF_south_sudan_df['realization time'] = ECMWF_south_sudan_df['date of prediction'] + (
            ECMWF_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_south_sudan_df['month'] = ECMWF_south_sudan_df['realization time'].dt.month
ECMWF_south_sudan_df['year'] = ECMWF_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
south_sudan_merged_df = ECMWF_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_ECMWF_merged.nc')


# Subset the Eastern East Africa Region ECMWF
ECMWF_eastern_east_africa = (ECMWF.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of ECMWF
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    ECMWF_eastern_east_africa, method='nearest')

# Calculate realized dates for ECMWF
ECMWF_eastern_east_africa_df = ECMWF_eastern_east_africa.to_dataframe().reset_index()
ECMWF_eastern_east_africa_df['lead_time'] = ECMWF_eastern_east_africa_df['lead_time'] - 0.5
ECMWF_eastern_east_africa_df['realization time'] = ECMWF_eastern_east_africa_df['date of prediction'] + (
            ECMWF_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_eastern_east_africa_df['month'] = ECMWF_eastern_east_africa_df['realization time'].dt.month
ECMWF_eastern_east_africa_df['year'] = ECMWF_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
eastern_east_africa_merged_df = ECMWF_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_ECMWF_merged.nc')

# Subset the Eastern Ukraine Region ECMWF
ECMWF_eastern_ukraine = (ECMWF.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    ECMWF_eastern_ukraine, method='nearest')

# Calculate realized dates for ECMWF
ECMWF_eastern_ukraine_df = ECMWF_eastern_ukraine.to_dataframe().reset_index()
ECMWF_eastern_ukraine_df['lead_time'] = ECMWF_eastern_ukraine_df['lead_time'] - 0.5
ECMWF_eastern_ukraine_df['realization time'] = ECMWF_eastern_ukraine_df['date of prediction'] + (
            ECMWF_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_eastern_ukraine_df['month'] = ECMWF_eastern_ukraine_df['realization time'].dt.month
ECMWF_eastern_ukraine_df['year'] = ECMWF_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
eastern_ukraine_merged_df = ECMWF_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_ECMWF_merged.nc')


# Subset the Southern Africa Region ECMWF
ECMWF_southern_africa = (ECMWF.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    ECMWF_southern_africa, method='nearest')

# Calculate realized dates for ECMWF
ECMWF_southern_africa_df = ECMWF_southern_africa.to_dataframe().reset_index()
ECMWF_southern_africa_df['lead_time'] = ECMWF_southern_africa_df['lead_time'] - 0.5
ECMWF_southern_africa_df['realization time'] = ECMWF_southern_africa_df['date of prediction'] + (
            ECMWF_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_southern_africa_df['month'] = ECMWF_southern_africa_df['realization time'].dt.month
ECMWF_southern_africa_df['year'] = ECMWF_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
southern_africa_merged_df = ECMWF_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_ECMWF_merged.nc')


# Subset the West Africa Region ECMWF
ECMWF_west_africa = (ECMWF.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(ECMWF_west_africa, method='nearest'))

# Calculate realized dates for ECMWF
ECMWF_west_africa_df = ECMWF_west_africa.to_dataframe().reset_index()
ECMWF_west_africa_df['lead_time'] = ECMWF_west_africa_df['lead_time'] - 0.5
ECMWF_west_africa_df['realization time'] = ECMWF_west_africa_df['date of prediction'] + (
            ECMWF_west_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_west_africa_df['month'] = ECMWF_west_africa_df['realization time'].dt.month
ECMWF_west_africa_df['year'] = ECMWF_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
west_africa_merged_df = ECMWF_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_ECMWF_merged.nc')


# Subset the Sri Lanka Region ECMWF
ECMWF_sri_lanka = (ECMWF.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(ECMWF_sri_lanka, method='nearest'))

# Calculate realized dates for ECMWF
ECMWF_sri_lanka_df = ECMWF_sri_lanka.to_dataframe().reset_index()
ECMWF_sri_lanka_df['lead_time'] = ECMWF_sri_lanka_df['lead_time'] - 0.5
ECMWF_sri_lanka_df['realization time'] = ECMWF_sri_lanka_df['date of prediction'] + (
            ECMWF_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_sri_lanka_df['month'] = ECMWF_sri_lanka_df['realization time'].dt.month
ECMWF_sri_lanka_df['year'] = ECMWF_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
sri_lanka_merged_df = ECMWF_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_ECMWF_merged.nc')


# Subset the Lake Victoria Basin Region ECMWF
ECMWF_lake_victoria_basin = (ECMWF.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(ECMWF_lake_victoria_basin, method='nearest'))

# Calculate realized dates for ECMWF
ECMWF_lake_victoria_basin_df = ECMWF_lake_victoria_basin.to_dataframe().reset_index()
ECMWF_lake_victoria_basin_df['lead_time'] = ECMWF_lake_victoria_basin_df['lead_time'] - 0.5
ECMWF_lake_victoria_basin_df['realization time'] = ECMWF_lake_victoria_basin_df['date of prediction'] + (
            ECMWF_lake_victoria_basin_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
ECMWF_lake_victoria_basin_df['month'] = ECMWF_lake_victoria_basin_df['realization time'].dt.month
ECMWF_lake_victoria_basin_df['year'] = ECMWF_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and ECMWF dataframe
lake_victoria_basin_merged_df = ECMWF_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_ECMWF_merged.nc')

In [ ]:
# Subset the South Sudan Region METEO
METEO_south_sudan = (METEO.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_south_sudan = chirps.sel(latitude=slice(3, 13), longitude=slice(24.5, 35.5)).interp_like(METEO_south_sudan,
                                                                                                method='nearest')

# Calculate realized dates for METEO
METEO_south_sudan_df = METEO_south_sudan.to_dataframe().reset_index()
METEO_south_sudan_df['lead_time'] = METEO_south_sudan_df['lead_time'] - 0.5
METEO_south_sudan_df['realization time'] = METEO_south_sudan_df['date of prediction'] + (
            METEO_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_south_sudan_df['month'] = METEO_south_sudan_df['realization time'].dt.month
METEO_south_sudan_df['year'] = METEO_south_sudan_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
south_sudan_merged_df = METEO_south_sudan_df.merge(chirps_south_sudan_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
south_sudan_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/south_sudan_METEO_merged.nc')


# Subset the Eastern East Africa Region METEO
METEO_eastern_east_africa = (METEO.sel(Y=slice(-3.5, 8), X=slice(38, 50))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of METEO
chirps_eastern_east_africa = chirps.sel(latitude=slice(-4, 8.5), longitude=slice(37.5, 50.5)).interp_like(
    METEO_eastern_east_africa, method='nearest')

# Calculate realized dates for METEO
METEO_eastern_east_africa_df = METEO_eastern_east_africa.to_dataframe().reset_index()
METEO_eastern_east_africa_df['lead_time'] = METEO_eastern_east_africa_df['lead_time'] - 0.5
METEO_eastern_east_africa_df['realization time'] = METEO_eastern_east_africa_df['date of prediction'] + (
            METEO_eastern_east_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_eastern_east_africa_df['month'] = METEO_eastern_east_africa_df['realization time'].dt.month
METEO_eastern_east_africa_df['year'] = METEO_eastern_east_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_east_africa_df = chirps_eastern_east_africa.to_dataframe().reset_index()
chirps_eastern_east_africa_df['month'] = chirps_eastern_east_africa_df['time'].dt.month
chirps_eastern_east_africa_df['year'] = chirps_eastern_east_africa_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
eastern_east_africa_merged_df = METEO_eastern_east_africa_df.merge(chirps_eastern_east_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_east_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_east_africa_METEO_merged.nc')

# Subset the Eastern Ukraine Region METEO
METEO_eastern_ukraine = (METEO.sel(Y=slice(45, 51), X=slice(31, 40))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_eastern_ukraine = chirps.sel(latitude=slice(44.5, 51.5), longitude=slice(30.5, 40.5)).interp_like(
    METEO_eastern_ukraine, method='nearest')

# Calculate realized dates for METEO
METEO_eastern_ukraine_df = METEO_eastern_ukraine.to_dataframe().reset_index()
METEO_eastern_ukraine_df['lead_time'] = METEO_eastern_ukraine_df['lead_time'] - 0.5
METEO_eastern_ukraine_df['realization time'] = METEO_eastern_ukraine_df['date of prediction'] + (
            METEO_eastern_ukraine_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_eastern_ukraine_df['month'] = METEO_eastern_ukraine_df['realization time'].dt.month
METEO_eastern_ukraine_df['year'] = METEO_eastern_ukraine_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_eastern_ukraine_df = chirps_eastern_ukraine.to_dataframe().reset_index()
chirps_eastern_ukraine_df['month'] = chirps_eastern_ukraine_df['time'].dt.month
chirps_eastern_ukraine_df['year'] = chirps_eastern_ukraine_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
eastern_ukraine_merged_df = METEO_eastern_ukraine_df.merge(chirps_eastern_ukraine_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
eastern_ukraine_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/eastern_ukraine_METEO_merged.nc')


# Subset the Southern Africa Region METEO
METEO_southern_africa = (METEO.sel(Y=slice(-23, -15), X=slice(25, 34))
                           .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_southern_africa = chirps.sel(latitude=slice(-23.5, -14.5), longitude=slice(24.5, 34.5)).interp_like(
    METEO_southern_africa, method='nearest')

# Calculate realized dates for METEO
METEO_southern_africa_df = METEO_southern_africa.to_dataframe().reset_index()
METEO_southern_africa_df['lead_time'] = METEO_southern_africa_df['lead_time'] - 0.5
METEO_southern_africa_df['realization time'] = METEO_southern_africa_df['date of prediction'] + (
            METEO_southern_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_southern_africa_df['month'] = METEO_southern_africa_df['realization time'].dt.month
METEO_southern_africa_df['year'] = METEO_southern_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_southern_africa_df = chirps_southern_africa.to_dataframe().reset_index()
chirps_southern_africa_df['month'] = chirps_southern_africa_df['time'].dt.month
chirps_southern_africa_df['year'] = chirps_southern_africa_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
southern_africa_merged_df = METEO_southern_africa_df.merge(chirps_southern_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
southern_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/southern_africa_METEO_merged.nc')


# Subset the West Africa Region METEO
METEO_west_africa = (METEO.sel(Y=slice(10, 13.5), X=slice(-10, 0))
                       .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_west_africa = (chirps.sel(latitude=slice(9.5, 14), longitude=slice(-10.5,
                                                                          0.5))  # One degree of resolution must be added for interpolation
                      .interp_like(METEO_west_africa, method='nearest'))

# Calculate realized dates for METEO
METEO_west_africa_df = METEO_west_africa.to_dataframe().reset_index()
METEO_west_africa_df['lead_time'] = METEO_west_africa_df['lead_time'] - 0.5
METEO_west_africa_df['realization time'] = METEO_west_africa_df['date of prediction'] + (
            METEO_west_africa_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_west_africa_df['month'] = METEO_west_africa_df['realization time'].dt.month
METEO_west_africa_df['year'] = METEO_west_africa_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_west_africa_df = chirps_west_africa.to_dataframe().reset_index()
chirps_west_africa_df['month'] = chirps_west_africa_df['time'].dt.month
chirps_west_africa_df['year'] = chirps_west_africa_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
west_africa_merged_df = METEO_west_africa_df.merge(chirps_west_africa_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
west_africa_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/west_africa_METEO_merged.nc')


# Subset the Sri Lanka Region METEO
METEO_sri_lanka = (METEO.sel(Y=slice(5.5, 10), X=slice(79.5, 82))
                     .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_sri_lanka = (chirps.sel(latitude=slice(5, 10.5),
                               longitude=slice(79, 82.5))  # One degree of resolution must be added for interpolation
                    .interp_like(METEO_sri_lanka, method='nearest'))

# Calculate realized dates for METEO
METEO_sri_lanka_df = METEO_sri_lanka.to_dataframe().reset_index()
METEO_sri_lanka_df['lead_time'] = METEO_sri_lanka_df['lead_time'] - 0.5
METEO_sri_lanka_df['realization time'] = METEO_sri_lanka_df['date of prediction'] + (
            METEO_sri_lanka_df['lead_time'] * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_sri_lanka_df['month'] = METEO_sri_lanka_df['realization time'].dt.month
METEO_sri_lanka_df['year'] = METEO_sri_lanka_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_sri_lanka_df = chirps_sri_lanka.to_dataframe().reset_index()
chirps_sri_lanka_df['month'] = chirps_sri_lanka_df['time'].dt.month
chirps_sri_lanka_df['year'] = chirps_sri_lanka_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
sri_lanka_merged_df = METEO_sri_lanka_df.merge(chirps_sri_lanka_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
sri_lanka_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/sri_lanka_METEO_merged.nc')


# Subset the Lake Victoria Basin Region METEO
METEO_lake_victoria_basin = (METEO.sel(Y=slice(-4, 1.5), X=slice(29, 36))
                               .rename(
    {'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date of prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of NMME
chirps_lake_victoria_basin = (chirps.sel(latitude=slice(-4.5, 2), longitude=slice(28.5,
                                                                                  36.5))  # One degree of resolution must be added for interpolation
                              .interp_like(METEO_lake_victoria_basin, method='nearest'))

# Calculate realized dates for METEO
METEO_lake_victoria_basin_df = METEO_lake_victoria_basin.to_dataframe().reset_index()
METEO_lake_victoria_basin_df['lead_time'] = METEO_lake_victoria_basin_df['lead_time'] - 0.5
METEO_lake_victoria_basin_df['realization time'] = METEO_lake_victoria_basin_df['date of prediction'] + (
            (METEO_lake_victoria_basin_df['lead_time'] - 0.5) * 30).astype('timedelta64[D]') # Lead 1 in CDS is same as Lead 0.5 in NMME
METEO_lake_victoria_basin_df['month'] = METEO_lake_victoria_basin_df['realization time'].dt.month
METEO_lake_victoria_basin_df['year'] = METEO_lake_victoria_basin_df['realization time'].dt.year

# Convert CHIRPS to dataframe
chirps_lake_victoria_basin_df = chirps_lake_victoria_basin.to_dataframe().reset_index()
chirps_lake_victoria_basin_df['month'] = chirps_lake_victoria_basin_df['time'].dt.month
chirps_lake_victoria_basin_df['year'] = chirps_lake_victoria_basin_df['time'].dt.year

# Merge CHIRPS and METEO dataframe
lake_victoria_basin_merged_df = METEO_lake_victoria_basin_df.merge(chirps_lake_victoria_basin_df, how='left', on=['month', 'year', 'latitude', 'longitude']).dropna(subset=['time'])

# Save to netCDF
lake_victoria_basin_merged_df.drop(['date of prediction', 'month', 'year', 'realization time'], axis=1).set_index(['lead_time', 'time', 'M', 'latitude', 'longitude']).to_xarray().to_netcdf('data/netCDF/lake_victoria_basin_METEO_merged.nc')